# Track 1 — LoRA Fine-Tuning Orchestrator

Thin orchestrator only. All real logic lives in `TASK1_finetuning_model/scripts/*.py`
(agent-editable, ordinary `.py` modules). Cells below just **sync code**, **install
deps**, and **call into those scripts**.

**Before running the sync cell:** after any local agent edit you MUST commit and push
(`git add -A && git commit -m ... && git push`) so the remote kernel pulls your
latest code. Re-running the sync cell picks up new edits.

In [ ]:
!git clone https://github.com/DevaNandanJS/Benchmarking-LLM-fine-tuning-vs-training-from-scratch-using-the-same-dataset.git llm_task 2>/dev/null || (cd llm_task && git pull)
%cd llm_task

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected - connect a Colab GPU kernel first"
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(props.total_memory / 1e9, 2))
print("torch:", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Phase 0 — Environment verification

The script below, `env_check.py`, is the Phase 0 environment verification — it also
serves as the "test local edit" that the sync cell is verified against (Phase 0 DoD).
Results land under `TASK1_finetuning_model/logs/` and are pulled back into the repo by the user.

In [ ]:
!python TASK1_finetuning_model/scripts/env_check.py

In [ ]:
!pip freeze > TASK1_finetuning_model/environment.txt
print("Wrote TASK1_finetuning_model/environment.txt - commit this back to the repo (reproducibility lock).")

## Phase 1 — Data Extraction & Cleaning

Runs `extract_text.py` which uses **pdfplumber** (primary) / **pypdf** (per-page fallback).

Outputs in `data/extracted/`:
- `document_clean.txt` — final cleaned text  
- `stats.json` — char / word / proxy-token counts  
- `extraction_manifest.json` — per-page extractor choice + quality scores  
- `hyphen_join_decisions.txt` — audit log of every line-break hyphen decision  
- `raw_pages/` — pre-clean text from both extractors, per page  

**After running:** open `data/extracted/document_clean.txt` and spot-check a few
random sections. Check `extraction_manifest.json` for any pages with `"garbled_flag": true`.

In [ ]:
!python TASK1_finetuning_model/scripts/extract_text.py

In [ ]:
# Quick sanity check: first & last 300 chars + stats summary
with open('data/extracted/document_clean.txt', encoding='utf-8') as f:
    text = f.read()
print('--- FIRST 300 CHARS ---')
print(text[:300])
print('\n--- LAST 300 CHARS ---')
print(text[-300:])

import json
stats = json.load(open('data/extracted/stats.json'))
print('\n--- STATS ---')
for k, v in stats.items():
    if k != 'note':
        print(f'  {k}: {v}')

# Show any garbled pages
manifest = json.load(open('data/extracted/extraction_manifest.json'))
garbled = [m for m in manifest if m['garbled_flag']]
if garbled:
    print(f'\n⚠ {len(garbled)} page(s) flagged as garbled:')
    for m in garbled:
        print(f"  page {m['page']:3d}  chosen={m['extractor_chosen']}  reason={m['reason']}")
        print(f"          non_ascii={m['final_scores']['non_ascii_ratio']:.1%}  "
              f"non_dict={m['final_scores']['non_dict_word_ratio']:.1%}")
else:
    print('\n✓ No pages flagged as garbled.')

## Phase 2 — Base Model & Tokenizer Selection

Runs `select_model.py` which:
1. Loads `HuggingFaceTB/SmolLM2-135M` tokenizer + model
2. Confirms/sets the pad token and logs the decision
3. Tokenizes `document_clean.txt` with the real tokenizer and updates `stats.json`
4. Extracts all `model.named_modules()` names → `configs/model_architecture.json`
   (Phase 4 reads this to set LoRA `target_modules` from verified names)
5. Runs fp16 memory math and records the quantization decision (no QLoRA needed)
6. Dumps `configs/run_phase2.json`

**After running:** check that:
- `data/extracted/stats.json` now has `exact_token_count_smollm2_135m`
- `configs/model_architecture.json` exists and lists `q_proj`/`v_proj`/etc.
- `configs/run_phase2.json` exists with `quantization_needed: false`
- All DoD assertions printed `✓`

In [ ]:
!python TASK1_finetuning_model/scripts/select_model.py

In [ ]:
# Phase 2 post-run inspection
import json

# 1. Updated token counts
stats = json.load(open('data/extracted/stats.json'))
print('--- TOKEN COUNTS ---')
print(f"  proxy_token_count_gpt2_tiktoken : {stats.get('proxy_token_count_gpt2_tiktoken')}")
print(f"  exact_token_count_smollm2_135m  : {stats.get('exact_token_count_smollm2_135m')}")

# 2. Run config summary
cfg = json.load(open('TASK1_finetuning_model/configs/run_phase2.json'))
print('\n--- RUN CONFIG SUMMARY ---')
for k in ('model_name', 'total_params', 'pad_token', 'quantization_needed',
          'fp16_weight_footprint_mb', 'vram_utilisation_weights_only_pct'):
    print(f"  {k}: {cfg.get(k)}")

# 3. Architecture spot-check — attention projection layers for Phase 4
arch = json.load(open('TASK1_finetuning_model/configs/model_architecture.json'))
attn_kw = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'c_attn')
attn = [n for n in arch['all_module_names'] if any(k in n for k in attn_kw)]
print(f"\n--- ATTENTION MODULES ({len(attn)} found — first 10) ---")
for name in attn[:10]:
    print(f"  {name}")


## Phase 3 - Dataset Construction (Chunking & Splitting)

Runs `build_dataset.py` which:
1. Loads SmolLM2-135M tokenizer and tokenizes `document_clean.txt`
2. Splits tokens into train (85%) / val (15%) by **contiguous holdout** (last 15% of token sequence)
3. Chunks train with stride=128 (50% overlap) for dense training samples
4. Chunks val with stride=256 (non-overlapping) for independent validation spans
5. Saves pure tensor dicts to `data/processed/finetune_train.pt` and `finetune_val.pt`
6. Writes `data/processed/dataset_stats.json` and `configs/split_strategy.md`

**After running:** check `dataset_stats.json` for exact chunk counts,
then spot-check a decoded sample chunk to confirm it reads like real document text.

In [ ]:
!python TASK1_finetuning_model/scripts/build_dataset.py

In [ ]:
# Phase 3 post-run inspection
import json, torch

# 1. Dataset stats
stats = json.load(open('data/processed/dataset_stats.json'))
print('--- DATASET STATS ---')
for k, v in stats.items():
    if k not in ('model_name', 'timestamp', 'note'):
        print(f'  {k}: {v}')

# 2. Tensor shapes (weights_only=True is clean -- pure tensor dict, no custom classes)
train_data = torch.load('data/processed/finetune_train.pt', weights_only=True)
val_data   = torch.load('data/processed/finetune_val.pt',   weights_only=True)
print('\n--- TENSOR SHAPES ---')
print('  train input_ids:', tuple(train_data['input_ids'].shape))
print('  val   input_ids:', tuple(val_data['input_ids'].shape))

# 3. Spot-check: decode first train and first val chunk
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-135M')
print('\n--- TRAIN CHUNK[0] (first 80 tokens decoded) ---')
print(tok.decode(train_data['input_ids'][0][:80]))
print('\n--- VAL CHUNK[0] (first 80 tokens decoded) ---')
print(tok.decode(val_data['input_ids'][0][:80]))


## Phase 4 — LoRA Configuration & Model Wrapping

Runs `wrap_lora.py` which:
1. Auto-detects device (CUDA or CPU) and selects dtype accordingly
   - **CUDA → fp16** (correct for T4/Turing SM 7.5; bf16 requires Ampere SM 8.0+)
   - **CPU → fp32** (fp16 `addmm`/`matmul` unsupported on CPU — avoids local dry-run errors)
2. Validates `target_modules=["q_proj", "v_proj"]` exist in `model.named_modules()` via
   PEFT's **substring** matching semantics (e.g., `"q_proj"` → `layers.N.self_attn.q_proj`)
3. Applies `LoraConfig(r=8, alpha=16, dropout=0.05)` — SmolLM2-135M Llama-style architecture
4. Captures `print_trainable_parameters()` output (authoritative GQA-aware count)
5. Authors sweep configs for Phase 5: `configs/run_phase4_r{4,8,16}.json`
6. Runs one forward/backward sanity pass on a batch from `finetune_train.pt`
7. Saves `configs/trainable_params.json` — required DoD deliverable

**After running:** check that:
- `configs/trainable_params.json` has `trainable_percentage` in the ~0.5–2% range
- `sanity_check_loss_finite: true` and loss is a reasonable float (~10 for a randomly initialised LoRA)
- All three sweep configs `run_phase4_r{4,8,16}.json` exist in `configs/`

**Phase 5 note flagged here:** Training loop MUST use `torch.cuda.amp.autocast` + `GradScaler`
— fp16 without loss scaling is a common NaN source on T4. See `wrap_lora.py` docstring.

In [ ]:
!python TASK1_finetuning_model/scripts/wrap_lora.py

In [ ]:
# Phase 4 post-run inspection
import json, os

params = json.load(open('TASK1_finetuning_model/configs/trainable_params.json'))
print('--- TRAINABLE PARAMS ---')
for k, v in params.items():
    if k != 'note':
        print(f'  {k}: {v}')

# Spot-check: trainable % should be <<10% for LoRA
pct = params.get('trainable_percentage', 0)
if 0 < pct < 10:
    print(f'\n  trainable_percentage={pct:.3f}% -- looks right for LoRA r=8')
else:
    print(f'\n  trainable_percentage={pct:.3f}% -- UNEXPECTED. Check target_modules.')

# Check sweep configs were written
for r in [4, 8, 16]:
    p = f'TASK1_finetuning_model/configs/run_phase4_r{r}.json'
    status = 'OK' if os.path.exists(p) else 'MISSING'
    print(f'  sweep config r={r}: {status} ({p})')

# Sanity loss
loss = params.get('sanity_check_loss')
finite = params.get('sanity_check_loss_finite')
print(f'\n  sanity_check_loss: {loss}  (finite: {finite})')


## Phase 5 — Training Loop & Execution

Runs `train.py` — a **manual training loop** (not HuggingFace `Trainer`).

**Why manual?** Every gradient step, scheduler tick, AMP scaler call, and
validation pass is explicit and explainable line by line. No `TrainingArguments`
magic to handwave in an interview.

**Sweep design:** three independent `--run` invocations, each producing its
own `logs/<run>/metrics.jsonl` and `checkpoints/best_val/<run>/` adapter.
After all three run, `eval/sweep_results.csv` has 3 rows.

**AMP (required):** `GradScaler` + `autocast(dtype=torch.float16)` are used.
fp16 without loss scaling → NaN gradients on T4. See `wrap_lora.py` docstring.

**Early stopping:** train stops if val loss rises for 2 consecutive epochs.
This is expected at this data scale — document the divergence step, don't chase it away.

**After running all 3 sweeps:** inspect `eval/sweep_results.csv` below to
identify the best run by `best_val_loss` — that run feeds Phase 6 and 7.

In [ ]:
# Sweep run 1 — LoRA r=4
!python TASK1_finetuning_model/scripts/train.py --run r4

In [ ]:
# Sweep run 2 — LoRA r=8 (plan default)
!python TASK1_finetuning_model/scripts/train.py --run r8

In [ ]:
# Sweep run 3 — LoRA r=16
!python TASK1_finetuning_model/scripts/train.py --run r16

In [ ]:
# Phase 5 — post-sweep inspection
import csv, os
csv_path = 'TASK1_finetuning_model/eval/sweep_results.csv'
if os.path.exists(csv_path):
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    print(f'Sweep results ({len(rows)} runs):')
    for r in rows:
        print(
            f"  {r['run_name']:6s}  "
            f"best_val={float(r['best_val_loss']):.4f}  "
            f"final_val={float(r['final_val_loss']):.4f}  "
            f"best_epoch={r['best_val_epoch']}"
        )
    best = min(rows, key=lambda x: float(x['best_val_loss']))
    print(f"\nBest run: {best['run_name']} (best_val_loss={float(best['best_val_loss']):.4f})")
else:
    print(f'sweep_results.csv not found at {csv_path} — run all 3 training cells first.')


## Phase 6 — Quantitative Evaluation

Runs `evaluate.py` on the best checkpoint from Phase 5.

**Outputs:**
- `eval/loss_curve.png` — train + val loss vs step (both curves, labelled)
- `eval/final_metrics.json` — CE loss, perplexity, **BPB**
- `eval/loss_curve_interpretation.md` — 3–5 sentence curve reading (edit after run)

**BPB validity:** val windows are non-overlapping (`val_stride=256=context_length`,
`val_overlap_pct=0.0` confirmed in `dataset_stats.json`). Every val token is
scored exactly once — BPB formula is valid.

**Byte alignment:** `evaluate.py` uses `return_offsets_mapping=True` to find
the exact character offset of the boundary token, avoiding the re-tokenization
drift that naive token-index slicing can introduce.

**After running:** edit `eval/loss_curve_interpretation.md` — the template is
populated with real statistics but the prose needs your human reading of the curve.

In [ ]:
# Phase 6 — evaluate best checkpoint (auto-picks from sweep_results.csv)
!python TASK1_finetuning_model/scripts/evaluate.py

In [ ]:
# Phase 6 — display final metrics and embed loss curve
import json
from IPython.display import Image, display

metrics = json.load(open('TASK1_finetuning_model/eval/final_metrics.json'))
print('--- FINAL METRICS ---')
for k, v in metrics.items():
    if k not in ('bpb_note', 'timestamp', 'checkpoint_dir'):
        print(f'  {k}: {v}')

print(f"\n  BPB = {metrics['bpb']}  (cross-track comparable metric)")
print(f"  Perplexity = {metrics['perplexity']}")

img_path = 'TASK1_finetuning_model/eval/loss_curve.png'
import os
if os.path.exists(img_path):
    display(Image(img_path))
else:
    print(f'loss_curve.png not found at {img_path}')


## Phase 7 — Qualitative Evaluation (Generation)

Runs `generate.py` — generates completions for 8 prompts drawn from the
**held-out validation region** (never seen during training).

**Prompt extraction:** deterministic (SEED=42), sentence-start based,
first ~60 tokens as prefix. Uses `return_offsets_mapping` for the same
byte-aligned boundary used in Phase 6.

**Two decoding modes per prompt:**
- Sampling: `temperature=0.8, top_p=0.9` (more varied, potentially more fluent)
- Greedy: deterministic (better for citing exact outputs in write-up)

**After running:** open `generations/finetuning_samples.md` and fill in the
`[TODO]` annotation for each sample — one sentence per prompt on whether
the completion is coherent, domain-relevant, and/or fluent.

In [ ]:
# Phase 7 — generate completions from best checkpoint
!python TASK1_finetuning_model/scripts/generate.py

In [ ]:
# Phase 7 — preview first 3 samples from finetuning_samples.md
import os
md_path = 'TASK1_finetuning_model/generations/finetuning_samples.md'
if os.path.exists(md_path):
    text = open(md_path, encoding='utf-8').read()
    # Print up to the 4th sample marker to preview without flooding output
    markers = [i for i, line in enumerate(text.splitlines()) if line.startswith('## Sample')]
    cutoff = text.splitlines()[markers[3]] if len(markers) >= 4 else text.splitlines()[-1]
    end_idx = text.find(cutoff) if len(markers) >= 4 else len(text)
    print(text[:end_idx])
    print(f'... ({len(markers)} total samples in {md_path})')
else:
    print(f'{md_path} not found — run generate.py first.')


## Phase 8 — Cross-Track Comparison Prep

Runs `stage_comparison.py` — copies Track 1 eval artifacts to `shared_eval/`
with namespace prefix (`track1_*`) so both tracks' results sit side by side.

**Outputs:**
- `shared_eval/finetuning_final_metrics.json` — BPB, perplexity, CE loss
- `shared_eval/finetuning_loss_curve.png` — loss curve for write-up
- `shared_eval/comparison_notes.md` — Track 1 section filled; Track 2 = TBD

**Phase 8 is fully completable once Track 1 is done.** The Track 2 fields
in `comparison_notes.md` remain TBD until Track 2 finishes — the script
stubs them rather than blocking.

In [ ]:
# Phase 8 — stage Track 1 artifacts for cross-track comparison
!python TASK1_finetuning_model/scripts/stage_comparison.py

In [ ]:
# Phase 8 — verify shared_eval/ contents
import os
shared = 'shared_eval'
if os.path.exists(shared):
    files = os.listdir(shared)
    print(f'shared_eval/ contents ({len(files)} files):')
    for f in sorted(files):
        size = os.path.getsize(os.path.join(shared, f))
        print(f'  {f}  ({size:,} bytes)')
else:
    print(f'{shared}/ not found — run stage_comparison.py first.')
